<a href="https://colab.research.google.com/github/silvia-jesus/MongoDB4SDM/blob/main/Pipeline_Reprodut%C3%ADvel_(GBIF_%2B_GEE_%2B_MongoDB_Atlas).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
# ============================================================
# Reproducible SDM Data Discovery Pipeline
# GBIF + Google Earth Engine + MongoDB Atlas (PyMongo)
# Study case: Carnivora in Parque Nacional das Emas (Brazil)
#
# Outputs:
# - MongoDB Atlas: runs, gbif_queries, gbif_occurrences, gee_layers, gee_exports
# - Google Drive: exported rasters (GeoTIFF)
#
# Core concept:
# - Every execution generates a run_id
# - All artifacts are linked to this run_id
# ============================================================

In [52]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Instalação de bibliotecas

In [53]:
!pip -q install pymongo geopandas shapely pyproj rasterio rtree tqdm
!pip -q install earthengine-api geemap
!pip -q install requests

Imports

In [54]:
import os
import json
import time
import uuid
import math
import requests
from datetime import datetime, timezone

import pandas as pd
import geopandas as gpd
from shapely.geometry import shape, mapping, box
from shapely.ops import unary_union

from tqdm import tqdm

from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.errors import BulkWriteError


import ee
import geemap

from getpass import getpass

Parâmetros mutáveis

In [55]:
password = getpass("MongoDB password: ")

MongoDB password: ··········


In [56]:
# ============================================================
# 1) PARAMETERS (mutable inputs)
# ============================================================

# ---- MongoDB Atlas ----
MONGODB_URI = (
    f"mongodb+srv://silviajesus_db_user:{password}"
    "@sdmcluster.tkqthsy.mongodb.net/"
    "?retryWrites=true&w=majority&appName=sdmcluster"
)
DB_NAME = "sdm_data_discovery"

# ---- Run metadata ----
RUN_TAG = "PNE_Carnivora_v1"
RUN_DESCRIPTION = "Reproducible data discovery for SDMs: GBIF + GEE + MongoDB Atlas"


In [57]:
# ============================
# AOI INPUT (3 MODES)
# ============================

# Modes:
# - "geojson_dict"  : AOI_GEOJSON is a dict already
# - "geojson_file"  : load GeoJSON from AOI_GEOJSON_PATH
# - "shapefile"     : load shapefile from SHAPEFILE_PATH

AOI_MODE = "geojson_file"  # "geojson_dict" | "geojson_file" | "shapefile"

# ---- Option 1: GeoJSON dict (in-memory) ----
AOI_GEOJSON = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {"name": "PNE_placeholder"},
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [-53.5, -18.5],
            [-53.5, -18.0],
            [-52.8, -18.0],
            [-52.8, -18.5],
            [-53.5, -18.5]
          ]
        ]
      }
    }
  ]
}

# ---- Option 2: GeoJSON file ----
AOI_GEOJSON_PATH = "/content/drive/MyDrive/PESQUISAS/MONGO_DB/DATA/AOI/PNE.geojson"  # upload this

# ---- Option 3: Shapefile ----
SHAPEFILE_PATH = "/content/aoi.shp"  # upload .shp + .shx + .dbf + .prj



In [58]:
# ---- Spatial preprocessing ----
BUFFER_KM = 10
TARGET_CRS = "ESRI:102033"  # Albers South America


In [59]:
# ============================
# LOAD AOI
# ============================
# 1) Load AOI as GeoDataFrame in WGS84
if AOI_MODE == "geojson_dict":
    _geojson = AOI_GEOJSON

elif AOI_MODE == "geojson_file":
    with open(AOI_GEOJSON_PATH, "r", encoding="utf-8") as f:
        _geojson = json.load(f)

elif AOI_MODE == "shapefile":
    gdf_aoi_wgs84 = gpd.read_file(SHAPEFILE_PATH)
    if gdf_aoi_wgs84.crs is None:
        raise ValueError("Shapefile has no CRS. Please upload the .prj or set CRS manually.")
    gdf_aoi_wgs84 = gdf_aoi_wgs84.to_crs("EPSG:4326")

else:
    raise ValueError("Invalid AOI_MODE. Use: geojson_dict | geojson_file | shapefile")

# If GeoJSON modes, convert to GeoDataFrame
if AOI_MODE in ["geojson_dict", "geojson_file"]:
    if "features" not in _geojson:
        raise ValueError("Invalid GeoJSON: missing 'features'. Must be a FeatureCollection.")
    gdf_aoi_wgs84 = gpd.GeoDataFrame.from_features(_geojson["features"], crs="EPSG:4326")

# 2) Clean geometries
gdf_aoi_wgs84 = gdf_aoi_wgs84[gdf_aoi_wgs84.geometry.notnull()].copy()
gdf_aoi_wgs84["geometry"] = gdf_aoi_wgs84.geometry.buffer(0)

if len(gdf_aoi_wgs84) == 0:
    raise ValueError("AOI is empty after cleaning (null/invalid geometries).")

# 3) Reproject to TARGET_CRS for buffering
gdf_aoi_proj = gdf_aoi_wgs84.to_crs(TARGET_CRS)

# 4) Buffer in meters
buffer_m = BUFFER_KM * 1000
gdf_aoi_proj["geometry"] = gdf_aoi_proj.geometry.buffer(buffer_m)

# 5) Dissolve into a single geometry (recommended for SDM AOI)
aoi_union_proj = gdf_aoi_proj.unary_union
gdf_aoi_proj = gpd.GeoDataFrame([{"geometry": aoi_union_proj}], crs=TARGET_CRS)

# 6) Back to WGS84 (for GBIF and GEE)
gdf_aoi_wgs84 = gdf_aoi_proj.to_crs("EPSG:4326")

# 7) Bounding box in WGS84
minx, miny, maxx, maxy = gdf_aoi_wgs84.total_bounds
aoi_bbox_wgs84 = (minx, miny, maxx, maxy)

# 8) Create bbox as GeoJSON (useful for logs and APIs)
bbox_geom = box(minx, miny, maxx, maxy)

aoi_bbox_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"type": "bbox"},
            "geometry": gpd.GeoSeries([bbox_geom], crs="EPSG:4326").__geo_interface__["features"][0]["geometry"]
        }
    ]
}


# ============================
# AOI SUMMARY (for provenance)
# ============================

print("AOI loaded successfully.")
print("AOI_MODE:", AOI_MODE)
print("TARGET_CRS:", TARGET_CRS)
print("BUFFER_KM:", BUFFER_KM)
print("AOI bounds (WGS84):", aoi_bbox_wgs84)
print("AOI area (km²) in projected CRS:", float(gdf_aoi_proj.area.iloc[0]) / 1e6)

AOI loaded successfully.
AOI_MODE: geojson_file
TARGET_CRS: ESRI:102033
BUFFER_KM: 10
AOI bounds (WGS84): (np.float64(-53.217840846253274), np.float64(-18.43555769696467), np.float64(-52.63831276141309), np.float64(-17.77747764577926))
AOI area (km²) in projected CRS: 3220.550891772061


/tmp/ipykernel_8667/1695638028.py:42: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  aoi_union_proj = gdf_aoi_proj.unary_union


In [60]:
# ---- GBIF ----
GBIF_TAXON_KEY = 732  # Carnivora (check in GBIF)
GBIF_COUNTRY = "BR"
GBIF_LIMIT = 300
GBIF_MAX_RECORDS = 20000  # safety cap
GBIF_HAS_COORD = True
GBIF_HAS_GEO_ISSUE = False

# Optional: restrict by bounding box derived from AOI
GBIF_USE_BBOX_FILTER = True

In [61]:
# ---- GEE ----
GEE_EXPORT_FOLDER_MONGODB = "SDM_PNE_Exports"
GEE_SCALE_DEFAULT = 30

In [62]:
# ---- Pixel-to-point demo ----
ENABLE_PIXEL_TO_POINT = True
PIXEL_SAMPLE_SCALE = 1000  # use 1000m for speed in demo


In [63]:
print("Parameters loaded.")

Parameters loaded.


Funções utilitárias

In [64]:
def utc_now():
    return datetime.now(timezone.utc)

def new_run_id():
    return str(uuid.uuid4())

def chunk_list(lst, chunk_size=1000):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i+chunk_size]

Conectar no MongoDB Atlas + criar índices

In [65]:
client = MongoClient(MONGODB_URI)
db = client[DB_NAME]

col_runs = db["runs"]
col_gbif_queries = db["gbif_queries"]
col_gbif_occ = db["gbif_occurrences"]
col_gee_layers = db["gee_layers"]
col_gee_exports = db["gee_exports"]

# ---- runs ----
col_runs.create_index([("run_id", ASCENDING)], unique=True)
col_runs.create_index([("created_at", DESCENDING)])


'created_at_-1'

In [66]:
# ---- gbif_queries ----
col_gbif_queries.create_index([("run_id", ASCENDING)])
col_gbif_queries.create_index([("created_at", DESCENDING)])

# ---- gbif_occurrences ----
col_gbif_occ.create_index([("run_id", ASCENDING)])
col_gbif_occ.create_index([("run_id", ASCENDING), ("gbif_key", ASCENDING)], unique=True)
col_gbif_occ.create_index([("taxonKey", ASCENDING)])
col_gbif_occ.create_index([("eventDate", DESCENDING)])

# 2dsphere for point geometry
col_gbif_occ.create_index([("location", "2dsphere")])

'location_2dsphere'

In [67]:
# ---- gee_layers ----
col_gee_layers.create_index([("run_id", ASCENDING)])
col_gee_layers.create_index([("dataset_id", ASCENDING)])

# ---- gee_exports ----
col_gee_exports.create_index([("run_id", ASCENDING)])
col_gee_exports.create_index([("task_id", ASCENDING)], unique=True)

print("MongoDB connected and indexes ensured.")

MongoDB connected and indexes ensured.


Criar o run_id e registrar a execução

In [68]:
run_id = new_run_id()

run_doc = {
    "run_id": run_id,
    "tag": RUN_TAG,
    "description": RUN_DESCRIPTION,
    "created_at": utc_now(),
    "status": "running",
    "parameters": {
        "BUFFER_KM": BUFFER_KM,
        "TARGET_CRS": TARGET_CRS,
        "GBIF_TAXON_KEY": GBIF_TAXON_KEY,
        "GBIF_COUNTRY": GBIF_COUNTRY,
        "GBIF_LIMIT": GBIF_LIMIT,
        "GBIF_MAX_RECORDS": GBIF_MAX_RECORDS,
        "GBIF_USE_BBOX_FILTER": GBIF_USE_BBOX_FILTER,
        "GEE_EXPORT_FOLDER": GEE_EXPORT_FOLDER_MONGODB,
        "GEE_SCALE_DEFAULT": GEE_SCALE_DEFAULT,
        "PIXEL_SAMPLE_SCALE": PIXEL_SAMPLE_SCALE,
        "AOI_MODE": AOI_MODE
    },
    "logs": []
}

col_runs.insert_one(run_doc)
print("run_id =", run_id)

run_id = f193ac9d-e642-43cc-91cd-bd8850e5ab1e


AOI

In [69]:
def load_aoi_gdf():
    mode = str(AOI_MODE).strip().lower()

    if mode == "geojson_dict":
        _geojson = AOI_GEOJSON

    elif mode == "geojson_file":
        with open(AOI_GEOJSON_PATH, "r", encoding="utf-8") as f:
            _geojson = json.load(f)

    elif mode == "shapefile":
        gdf = gpd.read_file(SHAPEFILE_PATH)
        if gdf.crs is None:
            raise ValueError("Shapefile has no CRS. Please upload the .prj or set CRS manually.")
        return gdf.to_crs("EPSG:4326")

    else:
        raise ValueError("Invalid AOI_MODE. Use: geojson_dict | geojson_file | shapefile")

    # GeoJSON -> GeoDataFrame
    if "features" not in _geojson:
        raise ValueError("Invalid GeoJSON: missing 'features'. Must be a FeatureCollection.")

    gdf = gpd.GeoDataFrame.from_features(_geojson["features"], crs="EPSG:4326")
    return gdf




aoi_gdf = load_aoi_gdf()
aoi_gdf = aoi_gdf.to_crs("EPSG:4326")

# dissolve
aoi_geom = unary_union(aoi_gdf.geometry)

# project to Albers for buffering
aoi_proj = gpd.GeoSeries([aoi_geom], crs="EPSG:4326").to_crs(TARGET_CRS).iloc[0]
buffer_m = BUFFER_KM * 1000
aoi_buffer_proj = aoi_proj.buffer(buffer_m)

# bbox in projected CRS
minx, miny, maxx, maxy = aoi_buffer_proj.bounds
bbox_proj = box(minx, miny, maxx, maxy)

# convert back to WGS84
aoi_buffer_wgs = gpd.GeoSeries([aoi_buffer_proj], crs=TARGET_CRS).to_crs("EPSG:4326").iloc[0]
bbox_wgs = gpd.GeoSeries([bbox_proj], crs=TARGET_CRS).to_crs("EPSG:4326").iloc[0]

aoi_doc = {
    "aoi_original_geojson": mapping(aoi_geom),
    "aoi_buffer_geojson": mapping(aoi_buffer_wgs),
    "bbox_geojson": mapping(bbox_wgs),
    "buffer_km": BUFFER_KM,
    "target_crs": TARGET_CRS
}

col_runs.update_one({"run_id": run_id}, {"$set": aoi_doc})
print("AOI stored in MongoDB (original + buffer + bbox).")

AOI stored in MongoDB (original + buffer + bbox).


GBIF: consulta paginada + registro em gbif_queries + ingestão em batches

In [70]:
GBIF_BASE = "https://api.gbif.org/v1/occurrence/search"

def gbif_build_params(offset=0):
    params = {
        "limit": GBIF_LIMIT,
        "offset": offset,
        "taxonKey": GBIF_TAXON_KEY,
        "country": GBIF_COUNTRY,
        "hasCoordinate": str(GBIF_HAS_COORD).lower(),
        "hasGeospatialIssue": str(GBIF_HAS_GEO_ISSUE).lower()
    }

    if GBIF_USE_BBOX_FILTER:
        # GBIF expects: minLon,minLat,maxLon,maxLat
        minx, miny, maxx, maxy = bbox_wgs.bounds
        params["geometry"] = f"POLYGON(({minx} {miny},{minx} {maxy},{maxx} {maxy},{maxx} {miny},{minx} {miny}))"

    return params

def gbif_request(params):
    r = requests.get(GBIF_BASE, params=params, timeout=60)
    return r

total_inserted = 0
offset = 0
end = False
page = 0

while not end:
    if total_inserted >= GBIF_MAX_RECORDS:
        print("Reached GBIF_MAX_RECORDS safety cap.")
        break

    params = gbif_build_params(offset=offset)
    r = gbif_request(params)

    query_doc = {
        "run_id": run_id,
        "created_at": utc_now(),
        "request_url": r.url,
        "params": params,
        "status_code": r.status_code
    }
    col_gbif_queries.insert_one(query_doc)

    if r.status_code != 200:
        col_runs.update_one({"run_id": run_id}, {"$push": {"logs": f"GBIF ERROR {r.status_code} at offset={offset}"}})
        break

    data = r.json()
    results = data.get("results", [])
    end = data.get("endOfRecords", True)

    if len(results) == 0:
        print("No more results.")
        break

    # Convert to MongoDB documents (Darwin Core partial)
    docs = []
    for occ in results:
        gbif_key = occ.get("key", None)
        lon = occ.get("decimalLongitude", None)
        lat = occ.get("decimalLatitude", None)

        if gbif_key is None or lon is None or lat is None:
            continue

        doc = {
            "run_id": run_id,
            "gbif_key": gbif_key,
            "taxonKey": occ.get("taxonKey"),
            "scientificName": occ.get("scientificName"),
            "species": occ.get("species"),
            "family": occ.get("family"),
            "order": occ.get("order"),
            "genus": occ.get("genus"),
            "basisOfRecord": occ.get("basisOfRecord"),
            "institutionCode": occ.get("institutionCode"),
            "collectionCode": occ.get("collectionCode"),
            "datasetKey": occ.get("datasetKey"),
            "publishingOrgKey": occ.get("publishingOrgKey"),
            "eventDate": occ.get("eventDate"),
            "year": occ.get("year"),
            "month": occ.get("month"),
            "day": occ.get("day"),
            "countryCode": occ.get("countryCode"),
            "stateProvince": occ.get("stateProvince"),
            "locality": occ.get("locality"),
            "coordinateUncertaintyInMeters": occ.get("coordinateUncertaintyInMeters"),
            "issues": occ.get("issues", []),
            "created_at": utc_now(),
            "location": {
                "type": "Point",
                "coordinates": [float(lon), float(lat)]
            },
            "raw": occ  # store raw for schema evolution
        }
        docs.append(doc)

    # Bulk insert with deduplication
    if len(docs) > 0:
        try:
            col_gbif_occ.insert_many(docs, ordered=False)
            inserted = len(docs)
        except BulkWriteError as bwe:
            # duplicates will be ignored due to unique index
            inserted = bwe.details.get("nInserted", 0)

        total_inserted += inserted

    page += 1
    offset += GBIF_LIMIT

    print(f"GBIF page={page} | fetched={len(results)} | inserted={inserted} | total_inserted={total_inserted}")

# Store summary
col_runs.update_one(
    {"run_id": run_id},
    {"$set": {
        "gbif_summary": {
            "total_inserted": total_inserted,
            "last_offset": offset,
            "finished_at": utc_now()
        }
    }}
)

print("GBIF ingestion done.")

GBIF page=1 | fetched=235 | inserted=235 | total_inserted=235
GBIF ingestion done.


Google Earth Engine: autenticar + definir camadas + registrar catálogo

In [71]:
!pip -q install earthengine-api
!earthengine authenticate --quiet

Authenticate: Limited support in Colab. Use ee.Authenticate() or --auth_mode=notebook instead.
Authenticate: Credentials already exist. Use --force to refresh.


In [72]:
# # Trigger the authentication flow
# ee.Authenticate()

In [73]:
# Initialize the library
ee.Initialize(project='sinuous-anvil-424317-g0')

In [74]:
# AOI geometry for EE
aoi_ee = ee.Geometry(mapping(aoi_buffer_wgs))

# Define layers (catalog)
gee_layers = [
    {
        "name": "ALOS_AW3D30_Elevation",
        "dataset_id": "JAXA/ALOS/AW3D30/V3_2",
        "asset_type": "ImageCollection",
        "band": "DSM",
        "type": "continuous",
        "scale": 30,
        "temporal_agg": "static"
    },
    {
        "name": "MODIS_MCD12Q1_LandCover",
        "dataset_id": "MODIS/061/MCD12Q1",
        "asset_type": "ImageCollection",   # <<< CORRIGIDO AQUI
        "band": "LC_Type1",
        "type": "categorical",
        "scale": 500,
        "temporal_agg": "annual",
        "year": 2020
    },
    {
        "name": "CHIRPS_Precip_Annual",
        "dataset_id": "UCSB-CHG/CHIRPS/DAILY",
        "asset_type": "ImageCollection",
        "band": "precipitation",
        "type": "continuous",
        "scale": 5500,
        "temporal_agg": "annual_sum",
        "year": 2020
    },
    {
        "name": "VIIRS_NighttimeLights",
        "dataset_id": "NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG",
        "asset_type": "ImageCollection",
        "band": "avg_rad",
        "type": "continuous",
        "scale": 500,
        "temporal_agg": "annual_mean",
        "year": 2020
    }
]

# Persist catalog to MongoDB
for lyr in gee_layers:
    lyr_doc = {
        "run_id": run_id,
        "created_at": utc_now(),
        **lyr
    }
    col_gee_layers.insert_one(lyr_doc)

print("GEE layer catalog stored in MongoDB.")

GEE layer catalog stored in MongoDB.


Exportar camadas do GEE para Drive + registrar tasks no MongoDB

In [75]:
def export_layer_to_drive(layer_def):
    name = layer_def["name"]
    dataset_id = layer_def["dataset_id"]
    band = layer_def["band"]
    scale = layer_def.get("scale", GEE_SCALE_DEFAULT)

    asset_type = layer_def.get("asset_type", "Image")

    # Load dataset correctly
    if asset_type == "Image":
        img = ee.Image(dataset_id)

    elif asset_type == "ImageCollection":
        img = ee.ImageCollection(dataset_id)

    else:
        raise ValueError(f"Unknown asset_type: {asset_type}")

    # Build image according to temporal aggregation
    temporal_agg = layer_def.get("temporal_agg", "static")

    if temporal_agg == "static":
        if asset_type == "Image":
            image = img.select(band)
        else:
            image = img.mosaic().select(band)

    elif temporal_agg == "annual":
        if asset_type != "ImageCollection":
            raise ValueError(f"{name}: temporal_agg='annual' requires ImageCollection")
        year = layer_def["year"]
        image = img.filter(ee.Filter.calendarRange(year, year, "year")).first().select(band)

    elif temporal_agg == "annual_sum":
        if asset_type != "ImageCollection":
            raise ValueError(f"{name}: temporal_agg='annual_sum' requires ImageCollection")
        year = layer_def["year"]
        image = img.filterDate(f"{year}-01-01", f"{year}-12-31").select(band).sum()

    elif temporal_agg == "annual_mean":
        if asset_type != "ImageCollection":
            raise ValueError(f"{name}: temporal_agg='annual_mean' requires ImageCollection")
        year = layer_def["year"]
        image = img.filterDate(f"{year}-01-01", f"{year}-12-31").select(band).mean()

    else:
        # fallback
        if asset_type == "Image":
            image = img.select(band)
        else:
            image = img.mosaic().select(band)

    # Clip
    image = image.clip(aoi_ee)

    file_prefix = f"{run_id}_{name}"

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=file_prefix,
        folder=GEE_EXPORT_FOLDER_MONGODB,
        fileNamePrefix=file_prefix,
        region=aoi_ee,   # se der erro, troque por aoi_ee.coordinates()
        scale=scale,
        maxPixels=1e13
    )
    task.start()

    export_doc = {
        "run_id": run_id,
        "created_at": utc_now(),
        "layer_name": name,
        "dataset_id": dataset_id,
        "asset_type": asset_type,
        "band": band,
        "scale": scale,
        "folder": GEE_EXPORT_FOLDER_MONGODB,
        "file_prefix": file_prefix,
        "task_id": task.id,
        "task_status": "SUBMITTED"
    }
    col_gee_exports.insert_one(export_doc)

    return task.id


task_ids = []
for lyr in gee_layers:
    tid = export_layer_to_drive(lyr)
    task_ids.append(tid)
    print("Submitted export task:", tid)

col_runs.update_one({"run_id": run_id}, {"$set": {"gee_export_tasks": task_ids}})
print("All exports submitted.")

Submitted export task: IP3WMIAPKS562HZZSKGO56C6
Submitted export task: GFWTWZ3IX4UQPAB5DM3BHW72
Submitted export task: SDWMEDHGH7H46SRTPWPC5XQX
Submitted export task: JXVN4VM2DW6B7SE3UEBTAB2M
All exports submitted.


Atualizar status das tasks

In [76]:
def refresh_tasks_status():
    exports = list(col_gee_exports.find({"run_id": run_id}))
    updated = 0

    for ex in exports:
        tid = ex["task_id"]
        st = ee.data.getTaskStatus(tid)[0]

        new_status = st.get("state", "UNKNOWN")
        col_gee_exports.update_one(
            {"task_id": tid},
            {"$set": {"task_status": new_status, "task_status_raw": st, "checked_at": utc_now()}}
        )
        updated += 1

    return updated

updated = refresh_tasks_status()
print("Updated task statuses:", updated)

Updated task statuses: 4


Pixel-to-point: amostrar valores ambientais nas ocorrências

In [77]:
if ENABLE_PIXEL_TO_POINT:
    print("Running pixel-to-point sampling (demo)...")

    # Fetch occurrences from MongoDB (limit for demo)
    occ_cursor = col_gbif_occ.find({"run_id": run_id}, {"gbif_key": 1, "location": 1}).limit(2000)
    occ_list = list(occ_cursor)

    # Convert to EE FeatureCollection
    feats = []
    for o in occ_list:
        lon, lat = o["location"]["coordinates"]
        feats.append(ee.Feature(ee.Geometry.Point([lon, lat]), {"gbif_key": o["gbif_key"]}))

    fc = ee.FeatureCollection(feats)

    # Use 2 layers for demo (fast)
    elev = ee.ImageCollection("JAXA/ALOS/AW3D30/V3_2").mosaic().select("DSM").clip(aoi_ee)
    viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
                .filterDate("2020-01-01", "2020-12-31") \
                .select("avg_rad").mean().clip(aoi_ee)

    stacked = elev.rename("elevation").addBands(viirs.rename("viirs"))

    sampled = stacked.sampleRegions(
        collection=fc,
        properties=["gbif_key"],
        scale=PIXEL_SAMPLE_SCALE,
        geometries=False
    )

    sampled_info = sampled.getInfo()
    rows = [f["properties"] for f in sampled_info["features"]]
    df_sample = pd.DataFrame(rows)

    print(df_sample.head())

    # Save sample output into MongoDB (optional)
    # Here we store it inside runs for compactness (demo)
    col_runs.update_one(
        {"run_id": run_id},
        {"$set": {"pixel_to_point_demo": {
            "n_points": len(df_sample),
            "variables": ["elevation", "viirs"],
            "scale": PIXEL_SAMPLE_SCALE,
            "created_at": utc_now()
        }}}
    )

    # Also save CSV to Drive for reviewer
    out_csv = f"/content/drive/MyDrive/PESQUISAS/MONGO_DB/{GEE_EXPORT_FOLDER_MONGODB}/{run_id}_pixel_to_point_demo.csv"
    df_sample.to_csv(out_csv, index=False)
    print("Saved pixel-to-point demo CSV to:", out_csv)

Running pixel-to-point sampling (demo)...
   elevation    gbif_key     viirs
0        851  5938675777  0.260000
1        851  6129861029  0.256250
2        851  6131598106  0.260000
3        847  6184996698  0.280000
4        769  6185058910  0.274583
Saved pixel-to-point demo CSV to: /content/drive/MyDrive/PESQUISAS/MONGO_DB/SDM_PNE_Exports/f193ac9d-e642-43cc-91cd-bd8850e5ab1e_pixel_to_point_demo.csv


Comparação entre runs (diff de ocorrências GBIF)

Aqui você consegue responder:

“Rodando de novo daqui 2 meses, o GBIF mudou?”

In [78]:
# Find previous runs with same tag
previous_runs = list(col_runs.find(
    {"tag": RUN_TAG, "run_id": {"$ne": run_id}},
    {"run_id": 1, "created_at": 1}
).sort("created_at", DESCENDING).limit(5))

print("Previous runs found:", len(previous_runs))
for r in previous_runs:
    print(r["run_id"], r["created_at"])

if len(previous_runs) > 0:
    prev_run_id = previous_runs[0]["run_id"]
    print("\nComparing current run with:", prev_run_id)

    # Get sets of gbif_keys (limit-safe: use cursor)
    curr_keys = set([d["gbif_key"] for d in col_gbif_occ.find({"run_id": run_id}, {"gbif_key": 1})])
    prev_keys = set([d["gbif_key"] for d in col_gbif_occ.find({"run_id": prev_run_id}, {"gbif_key": 1})])

    added = curr_keys - prev_keys
    removed = prev_keys - curr_keys

    diff_doc = {
        "compared_with": prev_run_id,
        "n_current": len(curr_keys),
        "n_previous": len(prev_keys),
        "n_added": len(added),
        "n_removed": len(removed),
        "created_at": utc_now()
    }

    col_runs.update_one({"run_id": run_id}, {"$set": {"gbif_diff": diff_doc}})

    print("Diff summary:", diff_doc)
else:
    print("No previous run to compare.")

Previous runs found: 5
48f1e6a2-07c9-46d3-b08d-2ceeb3c29b68 2026-06-23 18:08:12.788000
a4790851-9415-4b8f-a65b-94f575ab598e 2026-02-09 19:33:58.255000
91f12bc3-7ff7-402f-816e-5237f5d1da82 2026-02-09 19:11:35.684000
7da6377d-c399-4bd8-b88a-e9a33223201f 2026-02-09 18:56:11.197000
098e36e4-0697-4137-b1cb-8fa155741f32 2026-02-09 18:53:56.764000

Comparing current run with: 48f1e6a2-07c9-46d3-b08d-2ceeb3c29b68
Diff summary: {'compared_with': '48f1e6a2-07c9-46d3-b08d-2ceeb3c29b68', 'n_current': 235, 'n_previous': 235, 'n_added': 0, 'n_removed': 0, 'created_at': datetime.datetime(2026, 6, 23, 18, 23, 55, 909511, tzinfo=datetime.timezone.utc)}


**Demonstration queries**

In [79]:
print("---- Example queries ----")

# 1) Which layers are available for this run?
layers = list(col_gee_layers.find({"run_id": run_id}, {"_id": 0, "name": 1, "dataset_id": 1, "band": 1, "scale": 1}))
print("\nLayers for this run:")
for l in layers:
    print(l)

# 2) How many GBIF occurrences?
n_occ = col_gbif_occ.count_documents({"run_id": run_id})
print("\nGBIF occurrences for this run:", n_occ)

# 3) Export tasks status
exports = list(col_gee_exports.find({"run_id": run_id}, {"_id": 0, "layer_name": 1, "task_status": 1, "file_prefix": 1}))
print("\nExports:")
for e in exports:
    print(e)

---- Example queries ----

Layers for this run:
{'name': 'ALOS_AW3D30_Elevation', 'dataset_id': 'JAXA/ALOS/AW3D30/V3_2', 'band': 'DSM', 'scale': 30}
{'name': 'MODIS_MCD12Q1_LandCover', 'dataset_id': 'MODIS/061/MCD12Q1', 'band': 'LC_Type1', 'scale': 500}
{'name': 'CHIRPS_Precip_Annual', 'dataset_id': 'UCSB-CHG/CHIRPS/DAILY', 'band': 'precipitation', 'scale': 5500}
{'name': 'VIIRS_NighttimeLights', 'dataset_id': 'NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG', 'band': 'avg_rad', 'scale': 500}

GBIF occurrences for this run: 235

Exports:
{'layer_name': 'ALOS_AW3D30_Elevation', 'file_prefix': 'f193ac9d-e642-43cc-91cd-bd8850e5ab1e_ALOS_AW3D30_Elevation', 'task_status': 'RUNNING'}
{'layer_name': 'MODIS_MCD12Q1_LandCover', 'file_prefix': 'f193ac9d-e642-43cc-91cd-bd8850e5ab1e_MODIS_MCD12Q1_LandCover', 'task_status': 'RUNNING'}
{'layer_name': 'CHIRPS_Precip_Annual', 'file_prefix': 'f193ac9d-e642-43cc-91cd-bd8850e5ab1e_CHIRPS_Precip_Annual', 'task_status': 'READY'}
{'layer_name': 'VIIRS_NighttimeLights', 

Finalizar o run

In [80]:
col_runs.update_one(
    {"run_id": run_id},
    {"$set": {"status": "finished", "finished_at": utc_now()}}
)

print("Run finished:", run_id)

Run finished: f193ac9d-e642-43cc-91cd-bd8850e5ab1e


In [81]:
col_runs.update_one({"run_id": run_id}, {"$set": aoi_doc})

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff000000000000006b'), 'opTime': {'ts': Timestamp(1782239037, 2), 't': 107}, 'nModified': 0, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1782239037, 2), 'signature': {'hash': b'\xc3FI\xe8L\x99\x0c\xad;\x83~\x98\x1d\xf5\x19\x81m`\xc8\xba', 'keyId': 7610839824135618563}}, 'operationTime': Timestamp(1782239037, 2), 'updatedExisting': True}, acknowledged=True)

In [82]:
doc = col_runs.find_one({"run_id": run_id}, {"_id": 0, "run_id": 1, "aoi_original_geojson": 1, "aoi_buffer_geojson": 1, "bbox_geojson": 1})
doc

{'run_id': 'f193ac9d-e642-43cc-91cd-bd8850e5ab1e',
 'aoi_buffer_geojson': {'type': 'Polygon',
  'coordinates': [[[-53.21628755170339, -18.114110271323504],
    [-53.216303328299496, -18.11411226207526],
    [-53.21664483448167, -18.112055120232274],
    [-53.21688701696702, -18.110496082450542],
    [-53.217071086207284, -18.109224005793966],
    [-53.217840846253274, -18.099545694158238],
    [-53.21735949247153, -18.08985303467867],
    [-53.21563338513476, -18.080268590657074],
    [-53.21529661241398, -18.07890422090582],
    [-53.21521498111398, -18.0785587804645],
    [-53.215115023621195, -18.077915619767918],
    [-53.21330085824769, -18.069445361261888],
    [-53.21052732742934, -18.061174998831554],
    [-53.206822258104324, -18.053186993316146],
    [-53.20222274518521, -18.045560979553247],
    [-53.19677477606954, -18.038372973759614],
    [-53.190532767617704, -18.0316946173152],
    [-53.189794693468784, -18.030980846377894],
    [-53.18321014383712, -18.025188404024988]

In [85]:
print("\n" + "="*60)
print("PROVENANCE AUDIT REPORT")
print("="*60)

run = col_runs.find_one({"run_id": run_id})

print(f"\nRun ID: {run_id}")
print(f"Status: {run.get('status', 'N/A')}")
print(f"Created at: {run.get('created_at', 'N/A')}")

print("\nDocuments per collection:")

collections = {
    "runs": col_runs,
    "gbif_queries": col_gbif_queries,
    "gbif_occurrences": col_gbif_occ,
    "gee_layers": col_gee_layers,
    "gee_exports": col_gee_exports
}

total = 0

for name, col in collections.items():
    count = col.count_documents({"run_id": run_id})
    total += count
    print(f"- {name:<18}: {count}")

print("\nTotal linked documents:", total)

print("="*60)


PROVENANCE AUDIT REPORT

Run ID: f193ac9d-e642-43cc-91cd-bd8850e5ab1e
Status: finished
Created at: 2026-06-23 18:23:25.760000

Documents per collection:
- runs              : 1
- gbif_queries      : 1
- gbif_occurrences  : 235
- gee_layers        : 4
- gee_exports       : 4

Total linked documents: 245
